# Read-Level Benchmarking for Baleen (Real Data)

This notebook evaluates the pipeline's ability to correctly classify **individual reads** as modified or unmodified using real E. coli rRNA data with known modification sites.

## Ground Truth
- **Known modification sites**: E. coli 16S and 23S rRNA modifications from literature
- **Labeling rule**: At known mod sites → native reads = 1 (modified), IVT reads = 0 (unmodified)

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, f1_score
from sklearn.calibration import calibration_curve
import pandas as pd
import seaborn as sns
from collections import Counter

# Baleen imports
from baleen.eventalign import load_results, save_results
from baleen.eventalign._probability import (
    knn_ivt_purity,
    distance_to_ivt,
    mds_gmm,
)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# Try to find testdata
TESTDATA_PATHS = [
    Path.cwd().parent / 'testdata',
    Path.cwd() / 'testdata',
    Path('/Users/logan/Projects/py-baleen/testdata'),
]
TESTDATA_DIR = None
for p in TESTDATA_PATHS:
    if p.exists():
        TESTDATA_DIR = p
        break

if TESTDATA_DIR:
    print(f"Found testdata at: {TESTDATA_DIR}")
else:
    print("testdata not found - will use synthetic data for demonstration")

## 1. Known Modification Sites (E. coli 16S & 23S rRNA)

Ground truth modification sites from literature.

In [ ]:
# Position offset: pipeline uses 0-based, biological uses 1-based with flanking
POSITION_OFFSET = 3  # biological_pos = pipeline_pos + 3

KNOWN_MODIFICATIONS = {
    # --- Pseudouridine (Psi) ---
    ("ecoli16S", 516):  ("Ψ", "pseudouridine"),
    ("ecoli23S", 746):  ("Ψ", "pseudouridine"),
    ("ecoli23S", 955):  ("Ψ", "pseudouridine"),
    ("ecoli23S", 1911): ("Ψ", "pseudouridine"),
    ("ecoli23S", 1917): ("Ψ", "pseudouridine"),
    ("ecoli23S", 2457): ("Ψ", "pseudouridine"),
    ("ecoli23S", 2504): ("Ψ", "pseudouridine"),
    ("ecoli23S", 2580): ("Ψ", "pseudouridine"),
    ("ecoli23S", 2604): ("Ψ", "pseudouridine"),
    ("ecoli23S", 2605): ("Ψ", "pseudouridine"),
    # --- N2-methylguanosine (m2G) ---
    ("ecoli16S", 966):  ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1207): ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1516): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 1835): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 2445): ("m2G", "N2-methylguanosine"),
    # --- 5-methylcytidine (m5C) ---
    ("ecoli16S", 967):  ("m5C", "5-methylcytidine"),
    ("ecoli16S", 1407): ("m5C", "5-methylcytidine"),
    ("ecoli23S", 1962): ("m5C", "5-methylcytidine"),
    # --- 5-methyluridine (m5U) ---
    ("ecoli23S", 747):  ("m5U", "5-methyluridine"),
    ("ecoli23S", 1939): ("m5U", "5-methyluridine"),
    # --- N6,N6-dimethyladenosine (m6,6A) ---
    ("ecoli16S", 1518): ("m6,6A", "N6,N6-dimethyladenosine"),
    ("ecoli16S", 1519): ("m6,6A", "N6,N6-dimethyladenosine"),
    # --- N6-methyladenosine (m6A) ---
    ("ecoli23S", 1618): ("m6A", "N6-methyladenosine"),
    ("ecoli23S", 2030): ("m6A", "N6-methyladenosine"),
    # --- N7-methylguanosine (m7G) ---
    ("ecoli16S", 527):  ("m7G", "N7-methylguanosine"),
    ("ecoli23S", 2069): ("m7G", "N7-methylguanosine"),
    # --- Other modifications ---
    ("ecoli23S", 2498): ("Cm", "2′-O-methyl cytosine"),
    ("ecoli23S", 2449): ("D", "dihydrouridine"),
    ("ecoli23S", 2251): ("Gm", "2′-O-methyl guanine"),
    ("ecoli23S", 2552): ("Um", "2′-O-methyl uridine"),
    ("ecoli23S", 2501): ("ho5C", "5-hydroxycytidine"),
    ("ecoli23S", 745):  ("m1G", "1-methylguanosine"),
    ("ecoli23S", 2503): ("m2A", "2-methyladenosine"),
    ("ecoli23S", 1915): ("m3Ψ", "3-methylpseudouridine"),
    ("ecoli16S", 1498): ("m3U", "3-methyluridine"),
    ("ecoli16S", 1402): ("m4Cm", "N4,2′-O-dimethylcytidine"),
}

# Convert to pipeline positions (biological - offset)
KNOWN_MOD_PIPELINE = {
    (contig, bio_pos - POSITION_OFFSET): (mod_short, mod_full)
    for (contig, bio_pos), (mod_short, mod_full) in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_SET = set(KNOWN_MOD_PIPELINE.keys())

mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f"Total known modification sites: {len(KNOWN_MODIFICATIONS)}")
print(f"Modification types: {len(mod_counts)}\n")
for mod_type, count in mod_counts.most_common():
    full = next(v[1] for v in KNOWN_MODIFICATIONS.values() if v[0] == mod_type)
    print(f"  {mod_type:6s} ({full}): {count} sites")

## 2. Load Pipeline Results

Load pre-computed DTW results from the pipeline.

In [ ]:
# Look for pre-computed results
RESULTS_PATHS = [
    Path.cwd().parent / 'output' / 'pipeline_results.pkl',
    Path.cwd().parent / 'results' / 'pipeline_results.pkl',
    Path.cwd() / 'output' / 'pipeline_results.pkl',
]

results = None
metadata = None

for p in RESULTS_PATHS:
    if p.exists():
        print(f"Loading results from: {p}")
        results, metadata = load_results(p)
        break

if results is None:
    print("No pre-computed results found.")
    print("Run the pipeline first:")
    print("  baleen run --native-bam testdata/native_0.bam \\")
    print("              --native-fastq testdata/native_0/fastq/pass.fq.gz \\")
    print("              --native-blow5 testdata/native_0/blow5/nanopore.blow5 \\")
    print("              --ivt-bam testdata/control_0.bam \\")
    print("              --ivt-fastq testdata/control_0/fastq/pass.fq.gz \\")
    print("              --ivt-blow5 testdata/control_0/blow5/nanopore.blow5 \\")
    print("              --ref testdata/ref.fa \\")
    print("              --output-dir output/")
    print("\nFalling back to synthetic data...")
    USE_SYNTHETIC = True
else:
    USE_SYNTHETIC = False
    print(f"\nLoaded {len(results)} contigs")
    for contig, cr in results.items():
        print(f"  {contig}: {len(cr.positions)} positions")

## 3. Synthetic Data (Fallback)

Generate synthetic data if real results not available.

In [ ]:
def make_block_distance_matrix(n_native, n_ivt, within_native=1.0, within_ivt=1.0,
                               between=5.0, noise=0.1, seed=42):
    rng = np.random.RandomState(seed)
    n = n_native + n_ivt
    mat = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            if i < n_native and j < n_native:
                d = within_native
            elif i >= n_native and j >= n_native:
                d = within_ivt
            else:
                d = between
            d += rng.normal(0, noise)
            d = max(d, 0.0)
            mat[i, j] = d
            mat[j, i] = d
    return mat


def make_homogeneous_matrix(n_native, n_ivt, base_dist=1.0, noise=0.05, seed=42):
    rng = np.random.RandomState(seed)
    n = n_native + n_ivt
    mat = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            d = base_dist + rng.normal(0, noise)
            d = max(d, 0.0)
            mat[i, j] = d
            mat[j, i] = d
    return mat


if USE_SYNTHETIC:
    print("Generating synthetic data...")
    # This will be used when real data is not available

## 4. Extract Read-Level Labels and Scores

In [ ]:
def extract_read_level_data(results, known_mod_set):
    """Extract read-level scores and ground truth labels.
    
    Ground truth:
    - At known-mod positions: native reads = 1, IVT reads = 0
    - At other positions: all reads = 0 (treat as unmodified)
    """
    all_data = {
        'contig': [], 'position': [], 'read_name': [], 'is_native': [],
        'y_true': [], 'knn_score': [], 'dti_score': [], 'mds_score': [],
    }
    
    for contig, cr in results.items():
        for pos, pr in cr.positions.items():
            is_mod_pos = (contig, pos) in known_mod_set
            n_native = pr.n_native_reads
            n_ivt = pr.n_ivt_reads
            
            if n_native + n_ivt < 3:
                continue
            
            # Run algorithms
            try:
                knn_result = knn_ivt_purity(pr.distance_matrix, n_native, n_ivt)
                dti_result = distance_to_ivt(pr.distance_matrix, n_native, n_ivt)
                mds_result = mds_gmm(pr.distance_matrix, n_native, n_ivt)
            except Exception as e:
                continue
            
            # Native reads
            for i, name in enumerate(pr.native_read_names):
                all_data['contig'].append(contig)
                all_data['position'].append(pos)
                all_data['read_name'].append(name)
                all_data['is_native'].append(True)
                all_data['y_true'].append(1 if is_mod_pos else 0)
                all_data['knn_score'].append(knn_result.probabilities[i])
                all_data['dti_score'].append(dti_result.probabilities[i])
                all_data['mds_score'].append(mds_result.probabilities[i])
            
            # IVT reads
            for i, name in enumerate(pr.ivt_read_names):
                idx = n_native + i
                all_data['contig'].append(contig)
                all_data['position'].append(pos)
                all_data['read_name'].append(name)
                all_data['is_native'].append(False)
                all_data['y_true'].append(0)  # IVT always unmodified
                all_data['knn_score'].append(knn_result.probabilities[idx])
                all_data['dti_score'].append(dti_result.probabilities[idx])
                all_data['mds_score'].append(mds_result.probabilities[idx])
    
    return pd.DataFrame(all_data)


if not USE_SYNTHETIC and results is not None:
    df = extract_read_level_data(results, KNOWN_MOD_SET)
    print(f"Extracted {len(df)} read-level predictions")
    print(f"  Known modification sites: {df[df['y_true'] == 1].shape[0]} reads")
    print(f"  Unmodified positions: {df[df['y_true'] == 0].shape[0]} reads")
    df.head()

## 5. ROC Curves by Algorithm

In [ ]:
if not USE_SYNTHETIC and 'df' in dir() and len(df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    algorithms = [
        ('knn_score', 'kNN IVT-Purity (default)', '#2196F3'),
        ('dti_score', 'Distance-to-IVT', '#4CAF50'),
        ('mds_score', 'MDS + GMM', '#FF9800'),
    ]
    
    y_true = df['y_true'].values
    
    for ax, (col, label, color) in zip(axes, algorithms):
        scores = df[col].values
        
        fpr, tpr, _ = roc_curve(y_true, scores)
        roc_auc = auc(fpr, tpr)
        pr_auc = average_precision_score(y_true, scores)
        
        ax.plot(fpr, tpr, '-', color=color, lw=2, label=f'ROC (AUC={roc_auc:.3f})')
        ax.plot([0, 1], [0, 1], 'k--', lw=1)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'{label}\nPR-AUC={pr_auc:.3f}')
        ax.legend(loc='lower right')
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
    
    plt.suptitle('Read-Level ROC Curves (Known Modification Sites as Ground Truth)', y=1.02)
    plt.tight_layout()
    plt.savefig('read_level_real_roc.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No real data available. Run pipeline first to generate results.")

## 6. Performance by Modification Type

In [ ]:
if not USE_SYNTHETIC and 'df' in dir() and len(df) > 0:
    # Add modification type annotation
    df['mod_type'] = df.apply(
        lambda row: KNOWN_MOD_PIPELINE.get((row['contig'], row['position']), ("unmod", "unmodified"))[0],
        axis=1
    )
    
    # Compute AUC per modification type
    mod_metrics = []
    for mod_type in df['mod_type'].unique():
        if mod_type == 'unmod':
            continue
        subset = df[df['mod_type'] == mod_type]
        if len(subset) < 10 or len(subset['y_true'].unique()) < 2:
            continue
        
        y_true = subset['y_true'].values
        for col, algo in [('knn_score', 'kNN'), ('dti_score', 'DTI'), ('mds_score', 'MDS')]:
            try:
                roc = auc(*roc_curve(y_true, subset[col].values)[:2])
                pr = average_precision_score(y_true, subset[col].values)
            except:
                roc, pr = np.nan, np.nan
            mod_metrics.append({
                'mod_type': mod_type,
                'algorithm': algo,
                'roc_auc': roc,
                'pr_auc': pr,
                'n_reads': len(subset),
            })
    
    if mod_metrics:
        mod_df = pd.DataFrame(mod_metrics)
        
        fig, ax = plt.subplots(figsize=(12, 5))
        pivot = mod_df.pivot(index='mod_type', columns='algorithm', values='roc_auc')
        pivot.plot(kind='bar', ax=ax, color=['#2196F3', '#4CAF50', '#FF9800'])
        ax.set_xlabel('Modification Type')
        ax.set_ylabel('ROC-AUC')
        ax.set_title('Read-Level ROC-AUC by Modification Type')
        ax.legend(title='Algorithm')
        ax.set_ylim([0, 1.05])
        ax.axhline(y=0.5, color='gray', ls='--', lw=1)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig('read_level_by_mod_type.png', dpi=150, bbox_inches='tight')
        plt.show()

## 7. Score Distributions

In [ ]:
if not USE_SYNTHETIC and 'df' in dir() and len(df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    for ax, (col, label, color) in zip(axes, algorithms):
        modified = df[df['y_true'] == 1][col]
        unmodified = df[df['y_true'] == 0][col]
        
        ax.hist(unmodified, bins=30, alpha=0.6, label=f'Unmodified (n={len(unmodified)})',
                color='gray', density=True)
        ax.hist(modified, bins=30, alpha=0.6, label=f'Modified (n={len(modified)})',
                color='#FF5722', density=True)
        
        ax.set_xlabel('P(modified)')
        ax.set_ylabel('Density')
        ax.set_title(label)
        ax.legend()
        ax.set_xlim([-0.02, 1.02])
    
    plt.suptitle('Score Distributions (Modified vs Unmodified Reads)', y=1.02)
    plt.tight_layout()
    plt.savefig('read_level_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Calibration Analysis

In [ ]:
if not USE_SYNTHETIC and 'df' in dir() and len(df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    y_true = df['y_true'].values
    
    for ax, (col, label, color) in zip(axes, algorithms):
        scores = df[col].values
        
        try:
            prob_true, prob_pred = calibration_curve(y_true, scores, n_bins=10, strategy='uniform')
            ax.plot(prob_pred, prob_true, 's-', color=color, lw=2, label=label)
        except:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=ax.transAxes)
        
        ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfectly calibrated')
        ax.set_xlabel('Mean Predicted Probability')
        ax.set_ylabel('Fraction of Positives')
        ax.set_title(f'{label}')
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
    
    plt.suptitle('Probability Calibration Curves', y=1.02)
    plt.tight_layout()
    plt.savefig('read_level_calibration.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Summary Statistics

In [ ]:
if not USE_SYNTHETIC and 'df' in dir() and len(df) > 0:
    y_true = df['y_true'].values
    
    summary = []
    for col, algo in [('knn_score', 'kNN IVT-Purity'), ('dti_score', 'Distance-to-IVT'), ('mds_score', 'MDS+GMM')]:
        scores = df[col].values
        
        roc = auc(*roc_curve(y_true, scores)[:2])
        pr = average_precision_score(y_true, scores)
        
        # Best F1
        thresholds = np.linspace(0, 1, 100)
        f1s = [f1_score(y_true, scores > t, zero_division=0) for t in thresholds]
        best_f1 = max(f1s)
        best_thresh = thresholds[np.argmax(f1s)]
        
        summary.append({
            'Algorithm': algo,
            'ROC-AUC': roc,
            'PR-AUC': pr,
            'Best F1': best_f1,
            'Best Threshold': best_thresh,
        })
    
    summary_df = pd.DataFrame(summary)
    print("\n" + "="*70)
    print("READ-LEVEL BENCHMARK SUMMARY (Real Data)")
    print("="*70)
    print(f"\nTotal reads: {len(df)}")
    print(f"Modified reads (at known sites): {df['y_true'].sum()}")
    print(f"Unmodified reads: {(df['y_true'] == 0).sum()}")
    print(f"\nContigs: {df['contig'].nunique()}")
    print(f"Positions: {df.groupby(['contig', 'position']).ngroups}")
    print(f"Known mod sites in data: {df[df['y_true'] == 1].groupby(['contig', 'position']).ngroups}")
    print()
    print(summary_df.round(3).to_string(index=False))
else:
    print("No real data available for summary.")